# **Samuel Santiago Pinzón Reina**
# **Modelo Predictivo Supervivientes en el Titanic**

In [32]:
import pandas as pd

In [33]:
# Cargamos el dataframe
df = pd.read_csv('https://paste.c-net.org/GoodbyesRefill')

In [34]:
#Vistazo general al dataset
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# **Tratamiento de datos necesarios**

Eliminamos las columnas de las que no podamos obtener datos importantes:

In [35]:
df.drop(["Name", "Cabin", "PassengerId", "Ticket"], inplace = True, axis = 1)

#### Se reemplaza el genero por números para trabajar de mejor manera que con variables categoricas

In [36]:
df.infer_objects(copy=False)
df.replace({"male": 1, "female": 0}, inplace= True)
df.head(2)

<ipython-input-36-75f50aa0be60>:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({"male": 1, "female": 0}, inplace= True)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0.0,3,1,22.0,1,0,7.2500,S
1,1.0,1,0,38.0,1,0,71.2833,C


Lo mismo se hace con la columna del puerto de embarcación

In [37]:
df.replace({"C": 0, "Q": 1, "S": 2}, inplace = True)
df.fillna({"Embarked": 2}, inplace=True)

<ipython-input-37-3869815cd7d4>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({"C": 0, "Q": 1, "S": 2}, inplace = True)


#### Llenamos los datos vacíos en todas las columnas que podamos para que el modelo no tenga problemas a causa de tener filas vacías y tenga la mejor precisión posible.

Para la columna de edad:

In [38]:
df.fillna({"Age": df['Age'].mean()}, inplace=True)

Para la columna del puerto de embarcación:

In [39]:
df.fillna({"Embarked": 2}, inplace=True)

Llenamos ahora la columna de Fare con el precio más repetido

In [40]:
precio_moda = df['Fare'].mode()

In [41]:
df.fillna({"Fare": precio_moda[0]}, inplace = True)

# **Uso de Gradient Boosting Classifier para crear un modelo que prediga la cantidad de sobrevivientes en el accidente del Titanic**

El modelo seleccionado para esta tarea fue **Gradient Boost**, el cual me pareció que es un modelo bastante bueno para llegar a una mejor precisión, puesto que trabaja con arboles de decisión y cada arbol de decisión va corrijiendo, u optimizando, la predicción anterior mediante el calculo del error residual, así como que tambien va agregando un factor de aprendizaje. Lo que hace que se vaya ajustando con cada iteración y se vaya actualizando para que pueda hacer predicciones cada vez más precisas.

Importamos las librerías necesarias del modulo sklearn, el cual nos ayuda a separar los datos de entrenamiento y crear el modelo

In [42]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

Separamos los datos entre los que conocemos y los que no

In [43]:
conocidos = df[ ~df["Survived"].isna() ]
desconocidos = df[ df["Survived"].isna() ]

Tomamos el 20% de los datos conocidos para hacer las pruebas primero en los datos que conocemos

In [44]:
muestra = 0.2

Por ahora solo usamos "conocidos" para entrenar el modelo, de los cuales obtenemos las variables ```X``` e ```y```. Las cuales tomarán datos del dataframe de esta forma:

>```X:``` No se toma en cuenta la columna de sobrevivientes porque es sobre las otras columnas que el modelo va a aprender para realizar ajustes y predicciones. Eso sí, esto en base a su resultado en ```y```.

>```y:``` Se selecciona la columna de sobrevivientes porque es la que nos interesa para la creación del modelo.

In [45]:
X = conocidos.drop("Survived", axis = 1)
y = conocidos["Survived"]

La función train_test_split nos ayuda a separar de manera aleatoria los datos que irán para el entrenamiento y los que iran para testeo:

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = muestra, random_state = 0)

Creamos el modelo usando ```GradientBoostingClassifier``` y le damos los siguientes parametros:
> ```n_estimators```: Numero de arboles a usar. Usaremos 100 árboles de decisión, el cual es el valor recomendado, con el que podemos evitar un sobreentrenamiento

> ```max_depth```: Profundidad máxima de los árboles.
Le damos el argumento 5 porque es la profundidad máxima que queremos, así evitamos nuevamente el sobreentrenamiento. Y también es con esta profundidad que obtenemos mejores resultados.

>```learning_rate```: Tasa de aprendiaje. Le damos un valor bajo, lo que significa que cada árbol tendrá un aprendiaje más lento, pero que produce mejores resultados.

> ```random_state```: Semilla para el generador de números aleatorios. Usaremos la seed 0 para que los datos aleatorios no se pierdan y se pueda replicar todas las veces.

In [47]:
modelo = GradientBoostingClassifier(
    n_estimators = 100,
    max_depth = 5,
    learning_rate = 0.1,
    random_state = 0)

Empezamos a entrenar nuestro modelo y le damos como argumento los valores aleatorios que se escogieron del dataframe con los datos conocidos.

Este modelo empezará a realizar muchas iteraciones las cuales constan de predicciones hechas por los arboles creados, para posteriormente calcular el error residual y poco a poco volviendose un modelo ajustado para hacer predicciones.

In [48]:
modelo.fit(X_train, y_train)

GradientBoostingClassifier(max_depth=5, random_state=0)

Calculamos la precisión de nuestro modelo con un método integrado. Le damos como argumentos los valores predestinados para testear el modelo y el nos devuelve la precisión que tuvo:

In [49]:
print(f"La precisión de este modelo es del {modelo.score(X_test, y_test)*100 : .3f}%")

La precisión de este modelo es del  85.475%


Como vemos, **tiene una precision de el 85%**, lo que significa una **mejoría con respecto al modelo de RandomForestClassifier**, con el que en otras pruebas se obtuvo un 79% de precisión

#### Ahora que hemos entrenado el modelo y visto que tiene mucha efectividad, procedemos a realizar predicciones para los datos que desconocemos.

Creamos una nueva variable ```X_pred``` la cual tendrá los datos del dataframe _desconocidos_, tendrá los datos de todas las columnas, menos de _Survived_

In [50]:
X_pred = desconocidos.drop("Survived", axis = 1)

Usando el método ```predict()``` realizamos las predicciones en base a los datos almacenados en ```X_pred```para que el modelo prediga los sobrevivientes

In [51]:
y_pred = modelo.predict(X_pred)

Observamos las predicciones:

In [52]:
y_pred

array([0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 1., 1., 0.,
       0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 1., 1., 0., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 1., 1., 0.,
       0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1., 1., 1., 0.,
       0., 1., 1., 0., 0., 0., 1., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0.,
       0., 1., 1., 1., 1., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 0., 0., 1., 1.,
       1., 1., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
       1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 1., 0., 0.,
       0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 1., 0.,
       0., 1., 0., 0., 0., 1., 1., 0., 1., 1., 0., 0., 1., 0., 1., 0., 1.,
       0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 1., 1., 0., 1.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 1.,
       0., 1., 0., 1., 0.

Guardamos los valores de la predicción en el dataframe original, para reemplaar los valores nulos:

In [53]:
df.loc[desconocidos.index, "Survived"] = y_pred

Vemos como quedó ahora la **distribución de personas que murieron y que vivieron**:

In [54]:
df["Survived"].value_counts()

,count
Survived,
0.0,824
1.0,485


Y esta predicción tiene sentido, dado a que en la realidad hubo una distribución similar entre [supervivientes y personas que perdieron la vida](https://www.google.com/url?sa=t&rct=j&q=&esrc=s&source=web&cd=&cad=rja&uact=8&ved=2ahUKEwjVpNfdtoiNAxWVRTABHXFrFuQQFnoECBkQAw&url=https%3A%2F%2Fhistoria.nationalgeographic.com.es%2Fa%2Fvidas-truncadas-titanic_11387%23%3A~%3Atext%3DCuando%2520el%2520Titanic%2520se%2520hundi%25C3%25B3%2Cllegaron%2520a%2520ser%2520bastante%2520famosos.%26text%3DAbel%2520G.M.&usg=AOvVaw10LsrWmYETrqNioMSSF1hI&opi=89978449). Por lo que podemos decir que el modelo es funcional, debido a que refleja lo que pasó en la vida real con sus predicciones.